#**Laboratorio Práctico 05: Pandas II - Limpieza Avanzada, Merge y Exportación**


# **Caso de Negocio: Consolidación de Planilla y Evaluación de Desempeño**

El departamento de Recursos Humanos le ha enviado dos fuentes de datos distintas: un
reporte con los datos base de los empleados (con errores de tipeo y formatos incorrectos) y un
reporte con las evaluaciones de desempeño del último semestre. Su objetivo como Ingeniero
de Datos es limpiar la información, cruzar ambos reportes, calcular los bonos correspondientes
mediante lógicas avanzadas y exportar el archivo definitivo listo para que Finanzas realice los
pagos.

# **Parte 1: Creación de los Datos y Limpieza Avanzada**


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Tabla 1: Base de Empleados (Con errores intencionales)
df_empleados = pd.DataFrame({
"id_emp": ["E001", " E002 ", "e003", "E004"],
"nombre_completo": [" ana perez", "LUIS GOMEZ", "carlos ruiz", "maria diaz "],
"salario_base": ["USD 1200", "USD 1500", "USD 900", "USD 2000"]
})

In [ ]:
# Tabla 2: Evaluaciones de Desempeño
df_evaluaciones = pd.DataFrame({
"id_empleado": ["E001", "E002", "E003", "E009"], # Nota: E009 no existe en la base principal
"calificacion": ["A", "B", "A", "C"]
})

Misiones:
1. Limpie la columna id_emp del df_empleados. Utilice .apply(lambda x: ...) o los métodos
encadenados de .str para eliminar los espacios en blanco a los extremos y convertir
todas las letras a mayúsculas.
2. Estandarice la columna nombre_completo para que no tenga espacios sueltos a los lados
y utilice el formato de Título (Ej. "Ana Perez").
3. Limpie la columna salario_base utilizando .apply() y una función lambda para quitar el
texto "USD " y convertir el valor resultante a un número decimal (float).

In [ ]:
display(df_empleados)
display(df_evaluaciones)


,id_emp,nombre_completo,salario_base
0,E001,ana perez,USD 1200
1,E002,LUIS GOMEZ,USD 1500
2,e003,carlos ruiz,USD 900
3,E004,maria diaz,USD 2000


,id_empleado,calificacion
0,E001,A
1,E002,B
2,E003,A
3,E009,C


In [ ]:
# 1. Limpie la columna id_emp del df_empleados. Utilice .apply(lambda x: ...)
# o los métodos encadenados de .str para eliminar los espacios en blanco
# a los extremos y convertir todas las letras a mayúsculas.

df_empleados ["id_emp"] = df_empleados["id_emp"].apply(lambda x: x.strip().upper())

# 2. Estandarice la columna nombre_completo para que no tenga espacios sueltos a los lados y
#utilice el formato de Título (Ej. "Ana Perez").

df_empleados["nombre_completo"] = df_empleados["nombre_completo"].str.strip().str.title()

# 3. Limpie la columna salario_base utilizando .apply() y una función lambda para quitar
# el texto "USD " y convertir el valor resultante a un número decimal (float).

df_empleados["salario_base"] = df_empleados["salario_base"].apply(lambda x: float(x.replace("USD","")))



In [ ]:
display(df_empleados)


,id_emp,nombre_completo,salario_base
0,E001,Ana Perez,1200.0
1,E002,Luis Gomez,1500.0
2,E003,Carlos Ruiz,900.0
3,E004,Maria Diaz,2000.0


#**Parte 2: Cruce de Tablas (Merge)**

Ahora debe unir la información de desempeño a la base principal de empleados.
Misiones:
1. Utilice pd.merge() para realizar un cruce tipo LEFT JOIN donde df_empleados sea la
tabla "mandante" (a la izquierda). Queremos mantener a todos los empleados de la base,
tengan o no evaluación.
2. Dado que las llaves se llaman distinto (id_emp en la primera tabla vs id_empleado en la
segunda), utilice los parámetros left_on y right_on.
3. Guarde la tabla cruzada resultante en una variable llamada df_consolidado.
4. Notará que el cruce generó columnas redundantes. Elimine la columna id_empleado que
provino de la tabla de la derecha utilizando .drop(..., axis=1).

In [ ]:
#Unir la información de desempeño a la base principal de empleados

# LEFT JOIN
# Mantener a todos los empleados de la base, tengan o no evaluación.
# Dado que las llaves se llaman distinto (id_emp en la primera tabla vs id_empleado en la segunda),
    # utilice los parámetros left_on y right_on.

df_consolidado = pd.merge(df_empleados, df_evaluaciones,
                        left_on = "id_emp", right_on = "id_empleado",
                        how = "left")

display(df_consolidado)

,id_emp,nombre_completo,salario_base,id_empleado,calificacion
0,E001,Ana Perez,1200.0,E001,A
1,E002,Luis Gomez,1500.0,E002,B
2,E003,Carlos Ruiz,900.0,E003,A
3,E004,Maria Diaz,2000.0,NaN,NaN


In [ ]:
# Guardar la tabla cruzada resultante en una variable llamada df_consolidado y
# Eliminar columnas redundantes con .drop(..., axis = 1)

df_consolidado = df_consolidado.drop("id_empleado", axis=1)
display(df_consolidado)


,id_emp,nombre_completo,salario_base,calificacion
0,E001,Ana Perez,1200.0,A
1,E002,Luis Gomez,1500.0,B
2,E003,Carlos Ruiz,900.0,A
3,E004,Maria Diaz,2000.0,NaN


# **Parte 3: Lógica Condicional Multivariable (.apply con def)**


Finanzas ha determinado la siguiente regla para asignar los bonos de desempeño a la planilla
cruzada.
Misiones:
1. Cree una función en Python (utilizando def) llamada calcular_bono(fila) que reciba una
fila completa de la tabla.
2. La lógica interna de la función debe ser:

    ○ Si la calificación es "A", el bono será equivalente al 20% del salario_base.

    ○ Si la calificación es "B", el bono será equivalente al 10% del salario_base.

    ○ Si la calificación es nula o cualquier otra letra, el bono será 0.

3. Aplique su función al df_consolidado utilizando el método .apply(..., axis=1) para crear
una nueva columna llamada bono_asignado. (Recuerde que axis=1 es vital para que la
función pueda leer múltiples columnas de la misma fila).
4. Cree una columna final llamada salario_neto que sume el salario base más el bono
asignado.

In [ ]:
# Crenado la función para calcular el bono dadas ciertas condiciones
def calcular_bono(fila):
  calificacion = fila["calificacion"]
  salario = fila["salario_base"]

  if calificacion == "A":
    return fila ["salario_base"] * 0.20
  elif calificacion == "B":
    return fila ["salario_base"] * 0.10
  else:
    return 0.0

In [ ]:
# Aplicando la función de calcular_bono a df_consolidado
df_consolidado ["bono_calculado"] = df_consolidado.apply(calcular_bono, axis = 1)

In [ ]:
display(df_consolidado)

,id_emp,nombre_completo,salario_base,calificacion,bono_calculado
0,E001,Ana Perez,1200.0,A,240.0
1,E002,Luis Gomez,1500.0,B,150.0
2,E003,Carlos Ruiz,900.0,A,180.0
3,E004,Maria Diaz,2000.0,NaN,0.0


# **Parte 4: Fase LOAD (Exportaciones al Sistema)**


El pipeline ETL no termina hasta que los datos no son cargados a su destino final.
Misiones:
1. Exporte el DataFrame final df_consolidado a un archivo plano llamado
"planilla_final_2026.csv".
2. Asegúrese de incluir el parámetro index=False para que el sistema financiero no reciba
una columna extra de números de índice sin sentido.
3. (Reto Adicional) Exporte el mismo DataFrame a un archivo de Excel (.xlsx) en una
pestaña que se llame "Planilla_Neta".

In [ ]:
# Exportar dataframe final a un archivo llamado "planilla_final_2026.csv"

df_consolidado.to_csv("planilla_final_2026.csv", index = False)

# Exportar mismo dataframe a una pestaña llamada "Planilla_Neta"
df_consolidado.to_excel("planilla_final_2026.xlsx", sheet_name = "Plantilla_Neta", index = False)




Archivos generados: planilla_final_2026.csv y planilla_final_2026.xlsx
